# 10. Container Security & Image Hardening

## Background

Containers provide process isolation, but misconfigured containers routinely become lateral movement pivot points. Common misconfigurations: running as root inside the container, mounting the host Docker socket (instant host escape), overly permissive seccomp/AppArmor profiles, and basing images on full OS distributions that carry hundreds of vulnerable packages.

Container security spans two phases: **build-time** (Dockerfile linting, base image choice, layer hygiene) and **runtime** (image scanning, runtime policy, syscall filtering). Tools like Trivy and Grype scan images for CVEs. Hadolint lints Dockerfiles. Falco detects anomalous runtime behavior.

## What You'll Learn

- Dockerfile security anti-patterns and their attack surface
- Building a Dockerfile linter that catches common misconfigurations
- Image layer analysis: why layer hygiene matters for secrets
- Least-privilege container security model: user, capabilities, read-only filesystem

## Why This Matters

ML inference containers often run with elevated privileges 'for GPU access.' This is a misconception — NVIDIA Container Toolkit grants GPU access without requiring root inside the container. A containerized LLM service running as root with a leaked API key is one privilege escalation away from full host compromise.


## Dockerfile Security Linter

In [ ]:
import re
from dataclasses import dataclass
from typing import List, Tuple

@dataclass
class DockerfileLintFinding:
    line_number: int
    rule_id: str
    severity: str
    message: str
    line_content: str

DOCKERFILE_RULES = [
    ('DL3002', 'HIGH',
     r'^USER root$|^USER 0$',
     'Last USER instruction is root — container runs as root'),
    ('DL3006', 'HIGH',
     r'^FROM .+:latest',
     'Always tag the version; "latest" can silently pull breaking changes'),
    ('DL3007', 'MEDIUM',
     r'^FROM [^:@]+$',
     'Using image without specifying a tag — pin a specific version'),
    ('DL3025', 'HIGH',
     r'--privileged',
     'Container with --privileged has full host capabilities'),
    ('DL3008', 'MEDIUM',
     r'apt-get install(?!.*--no-install-recommends)',
     'apt-get install without --no-install-recommends bloats image'),
    ('SC0001', 'CRITICAL',
     r'(?i)password|api_key|secret|token',
     'Possible secret in Dockerfile — use build secrets or env at runtime'),
    ('DL4006', 'HIGH',
     r'curl.*\|.*sh|wget.*\|.*sh',
     'Piping curl/wget to shell — supply chain risk'),
    ('DL3020', 'MEDIUM',
     r'^ADD .+',
     'Use COPY instead of ADD; ADD silently extracts archives and fetches URLs'),
    ('DL3009', 'LOW',
     r'apt-get update(?!.*&&)',
     'apt-get update without combining with install in same RUN layer'),
    ('DL3042', 'HIGH',
     r'-v /var/run/docker.sock:/var/run/docker.sock',
     'Docker socket mounted — container can escape to host'),
]

def lint_dockerfile(content: str) -> List[DockerfileLintFinding]:
    findings = []
    lines = content.splitlines()

    # Check for missing USER instruction (running as root by default)
    has_non_root_user = any(
        re.match(r'^USER (?!root|0)', line.strip())
        for line in lines if line.strip().startswith('USER')
    )
    if not has_non_root_user:
        findings.append(DockerfileLintFinding(
            line_number=0, rule_id='DL3002', severity='HIGH',
            message='No non-root USER instruction found — defaults to root',
            line_content='(missing)'
        ))

    for i, line in enumerate(lines, 1):
        stripped = line.strip()
        if stripped.startswith('#'): continue

        for rule_id, severity, pattern, message in DOCKERFILE_RULES:
            if re.search(pattern, stripped):
                findings.append(DockerfileLintFinding(
                    line_number=i, rule_id=rule_id, severity=severity,
                    message=message, line_content=stripped[:80]
                ))
    return findings

INSECURE_DOCKERFILE = '''
FROM python:latest
ENV OPENAI_API_KEY=sk-test1234567890abcdef
RUN apt-get update
RUN apt-get install -y curl git vim
RUN curl https://get.example.com/install.sh | sh
ADD . /app
WORKDIR /app
RUN pip install -r requirements.txt
CMD ["python", "app.py"]
'''

findings = lint_dockerfile(INSECURE_DOCKERFILE)
print(f'Dockerfile lint: {len(findings)} findings\n')
for f in findings:
    loc = f'line {f.line_number}' if f.line_number else 'file'
    print(f'  [{f.severity}] {f.rule_id} ({loc}): {f.message}')


## Secure Dockerfile Template

In [ ]:
SECURE_DOCKERFILE = '''
# --- Build stage ---
FROM python:3.11-slim AS builder

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --no-install-recommends -r requirements.txt

# --- Runtime stage ---
FROM python:3.11-slim

# Create non-root user
RUN groupadd -r appgroup && useradd -r -g appgroup -s /sbin/nologin appuser

WORKDIR /app

# Copy only installed packages and app code
COPY --from=builder /usr/local/lib/python3.11/site-packages /usr/local/lib/python3.11/site-packages
COPY --chown=appuser:appgroup src/ .

# Drop privileges
USER appuser

# Security labels
LABEL org.opencontainers.image.source="https://github.com/my-org/llm-app"
LABEL security.non-root="true"
LABEL security.read-only-rootfs="true"

# API key via secret at runtime — never in image
# docker run --env-file .env myapp

EXPOSE 8080
ENTRYPOINT ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
'''

secure_findings = lint_dockerfile(SECURE_DOCKERFILE)
print(f'Secure Dockerfile lint: {len(secure_findings)} findings')
if secure_findings:
    for f in secure_findings:
        print(f'  [{f.severity}] {f.rule_id}: {f.message}')
else:
    print('  No issues found!')

print('\nSecurity improvements:')
improvements = [
    ('Multi-stage build', 'Eliminates build tools from runtime image'),
    ('python:3.11-slim', 'Pinned version, minimal base = fewer CVEs'),
    ('Non-root user', 'appuser: limits blast radius of container compromise'),
    ('--no-install-recommends', 'Reduces installed packages by ~60%'),
    ('COPY vs ADD', 'No hidden URL fetching or archive extraction'),
    ('Secrets at runtime', 'API key injected via env at run time, never baked in'),
]
for improvement, reason in improvements:
    print(f'  {improvement:30s} {reason}')
